### Data-Level Security Using Row filters and Column Masks
Unlike dynamic Views, where logic is defined in the view, this approach applies security rules directly to the table. These rules are enforced automatically whenever the table is queried.

####1. Set up the schema for the table

In [0]:
USE CATALOG demo;
USE SCHEMA data_security;

####2. Create the Table

In [0]:
CREATE OR REPLACE TABLE sales_secured(
    id INT,
    region STRING,
    email STRING,
    revenue INT
)

#### 3. Insert the records

In [0]:
INSERT INTO sales_secured
VALUES 
(1, 'UK', 'john.doe@gmail.com', 100), 
(2, 'US', 'jane.smith@yahoo.com', 200), 
(3, 'UK', 'bob.jones@hotmail.com', 300), 
(4, 'US', 'sara.williams@gmail.com', 400)


In [0]:
SELECT * FROM sales_secured;

####4. Define Row Filter Function

In [0]:
CREATE OR REPLACE FUNCTION fn_filter_region(region STRING)
RETURN
  is_account_group_member('admin-sg')
  OR (is_account_group_member('us-sg') AND region = 'US')
  OR (is_account_group_member('uk-sg') AND region = 'UK');

####5. Apply Row Filter to Table

In [0]:
ALTER TABLE sales_secured
SET ROW FILTER fn_filter_region ON (region);

In [0]:
SELECT * FROM sales_secured

####6. Define Column Mask Function

In [0]:
CREATE OR REPLACE FUNCTION fn_mask_email(email STRING)
RETURN
  CASE WHEN is_account_group_member('admin-sg') THEN email
       ELSE CONCAT(SUBSTR(email, 1, 1), '****@', split(email, '@')[1])
    END;

        

In [0]:
ALTER TABLE sales_secured
ALTER COLUMN email SET MASK fn_mask_email;


In [0]:
SELECT * FROM sales_secured